In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/christinezakaria/predictive-maintenance-data4/All_test_data.xlsx
/kaggle/input/datasets/christinezakaria/predictive-maintenance-data4/RUL_data.xlsx
/kaggle/input/datasets/christinezakaria/predictive-maintenance-data4/All_train_data (1).xlsx


Load multi-sheet Excel dataset and merge into a single DataFrame.
Each sheet contains different engines, so we offset unit_id to ensure uniqueness.
A new column "dataset" is added to track the source sheet.
Important: This step prepares raw data before cleaning, sorting, and labeling.

In [2]:
import pandas as pd

file_path = "/kaggle/input/datasets/christinezakaria/predictive-maintenance-data4/All_train_data (1).xlsx"

all_sheets = pd.read_excel(file_path, sheet_name=None)

frames = []
uid_offset = 0

for sheet_name, df in all_sheets.items():
    
    if "unit id" in df.columns:
        df = df.rename(columns={"unit id": "unit_id"})
    
    df = df.copy()
    
    df["unit_id"] = df["unit_id"] + uid_offset
    df["dataset"] = sheet_name
    
    uid_offset += df["unit_id"].max()   # safer
    
    frames.append(df)

df = pd.concat(frames, ignore_index=True)


df = df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)

print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nUnique engines:", df['unit_id'].nunique())

display(df.head())

Shape: (160359, 27)

Columns:
 Index(['unit_id', 'cycle', 'op_set_1', 'op_set_2', 'op_set_3', 's1', 's2',
       's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13',
       's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 'dataset'],
      dtype='object')

Unique engines: 709


,unit_id,cycle,op_set_1,op_set_2,op_set_3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,dataset
0,1,1,-0.0007,-0.0004,100,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,train_FD001(HPC Degradation)
1,1,2,0.0019,-0.0003,100,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,train_FD001(HPC Degradation)
2,1,3,-0.0043,0.0003,100,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,train_FD001(HPC Degradation)
3,1,4,0.0007,0.0000,100,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,train_FD001(HPC Degradation)
4,1,5,-0.0019,-0.0002,100,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,train_FD001(HPC Degradation)


1. Check for missing values in each column.
2. Display statistical summary of numerical features.
3. Analyze engine lifetimes (max cycles per unit).


In [3]:
print("Missing values:\n", df.isnull().sum())
print("\nStats:\n", df.describe())

engine_lengths = df.groupby("unit_id")["cycle"].max()

print("\nEngine lengths:")
print(engine_lengths.describe())

Missing values:
 unit_id     0
cycle       0
op_set_1    0
op_set_2    0
op_set_3    0
s1          0
s2          0
s3          0
s4          0
s5          0
s6          0
s7          0
s8          0
s9          0
s10         0
s11         0
s12         0
s13         0
s14         0
s15         0
s16         0
s17         0
s18         0
s19         0
s20         0
s21         0
dataset     0
dtype: int64

Stats:
              unit_id          cycle       op_set_1       op_set_2  \
count  160359.000000  160359.000000  160359.000000  160359.000000   
mean      599.577043     123.331338      17.211973       0.410004   
std       450.409236      83.538146      16.527988       0.367938   
min         1.000000       1.000000      -0.008700      -0.000600   
25%       196.000000      57.000000       0.001300       0.000200   
50%       481.000000     114.000000      19.998100       0.620000   
75%      1107.000000     173.000000      35.001500       0.840000   
max      1269.000000     543.00

Compute Remaining Useful Life (RUL) for each engine at each cycle.
RUL = (maximum cycle of the engine) - (current cycle).
Then optionally clip RUL to reduce extreme values (e.g., max 125)
to make the data more stable for training.
Create binary classification label based on prediction horizon:
label = 1 if RUL <= 30 (failure will happen soon)
label = 0 otherwise

In [4]:
max_cycle_per_engine = df.groupby("unit_id")["cycle"].max()
df = df.merge(max_cycle_per_engine.rename("max_cycle"), on="unit_id")

df["RUL"] = df["max_cycle"] - df["cycle"]
df = df.drop(columns=["max_cycle"])

df["RUL"] = df["RUL"].clip(upper=125)

# Classification label
threshold = 30
df["label"] = (df["RUL"] <= threshold).astype(int)


print(df[["unit_id", "cycle", "RUL", "label"]].head(10))


print("\nLabel distribution:")
print(df["label"].value_counts(normalize=True))

   unit_id  cycle  RUL  label
0        1      1  125      0
1        1      2  125      0
2        1      3  125      0
3        1      4  125      0
4        1      5  125      0
5        1      6  125      0
6        1      7  125      0
7        1      8  125      0
8        1      9  125      0
9        1     10  125      0

Label distribution:
label
0    0.862939
1    0.137061
Name: proportion, dtype: float64


windowing
Build sliding windows for time-series modeling.
For each engine, create sequences of length window_size using past cycles only.
Each window contains consecutive sensor readings (no future data).
The label (classification and RUL) is taken from the last timestep in the window,
meaning the model predicts the future based on past observations.
This ensures no data leakage and proper time-series learning.

In [5]:
import numpy as np

def build_sliding_windows(df, window_size=30, stride=1):

   
    feature_cols = [col for col in df.columns 
                    if col.startswith("s") or col.startswith("op")]

    X, y_cls, y_rul, engine_ids = [], [], [], []

    for uid, grp in df.groupby("unit_id"):
        grp = grp.sort_values("cycle")

        features = grp[feature_cols].values
        labels = grp["label"].values
        rul = grp["RUL"].values

        n_steps = len(grp)

        if n_steps < window_size:
            continue

        for start in range(0, n_steps - window_size + 1, stride):
            end = start + window_size

            X.append(features[start:end])
            y_cls.append(labels[end - 1])
            y_rul.append(rul[end - 1])
            engine_ids.append(uid)

    X = np.array(X, dtype=np.float32)
    y_cls = np.array(y_cls, dtype=np.int64)
    y_rul = np.array(y_rul, dtype=np.float32)
    engine_ids = np.array(engine_ids)

    print("X shape:", X.shape)
    print("y_cls shape:", y_cls.shape)

    return X, y_cls, y_rul, engine_ids

In [6]:
X_raw, y_cls, y_rul, engine_ids = build_sliding_windows(df)

print("X:", X_raw.shape)
print("y_cls:", y_cls.shape)
print("y_rul:", y_rul.shape)

X shape: (139798, 30, 24)
y_cls shape: (139798,)
X: (139798, 30, 24)
y_cls: (139798,)
y_rul: (139798,)


In [7]:
X_raw, y_cls, y_rul, engine_ids = build_sliding_windows(df)

print("X:", X_raw.shape)
print("y_cls:", y_cls.shape)
print("y_rul:", y_rul.shape)

# Check engine distribution
print("\nUnique engines:", len(set(engine_ids)))

# Check label balance
print("\nLabel distribution:")
print(pd.Series(y_cls).value_counts(normalize=True))

# Sanity check sample
print("\nFirst sample label:", y_cls[0])

X shape: (139798, 30, 24)
y_cls shape: (139798,)
X: (139798, 30, 24)
y_cls: (139798,)
y_rul: (139798,)

Unique engines: 709

Label distribution:
0    0.84278
1    0.15722
Name: proportion, dtype: float64

First sample label: 0


In [8]:
print(X_raw[0].shape)
print(y_cls[0])
print(y_rul[0])

(30, 24)
0
125.0


features eng
Sequence-level feature engineering.
Adds additional temporal features for each window:
- diff: difference between consecutive timesteps (captures change)
- rolling mean: smooths short-term noise
- trend: linear slope of the signal over the window
Final features = [original + diff + rolling + trend]
This increases feature dimension and helps capture temporal patterns.

In [9]:
import numpy as np

class SequenceFeatureExtractor:
    def __init__(self, window_size=30):
        self.window_size = window_size

    def transform(self, X):
       

        diff = np.diff(X, axis=1, prepend=X[:, :1, :])
        rolling = self._rolling_mean(X, k=3)
        trend = self._trend_fast(X)

        
        X_new = np.concatenate([X, diff, rolling, trend], axis=2)

        return X_new.astype(np.float32)

    def _rolling_mean(self, X, k=3):
        N, W, F = X.shape
        out = np.zeros_like(X)

        for t in range(W):
            start = max(0, t - k + 1)
            out[:, t, :] = X[:, start:t+1, :].mean(axis=1)

        return out

    def _trend_fast(self, X):
      
        slope = (X[:, -1, :] - X[:, 0, :]) / X.shape[1]   # (N, F)

     
        slope = slope[:, np.newaxis, :]

       
        trend = np.repeat(slope, X.shape[1], axis=1)

        return trend

In [10]:
extractor = SequenceFeatureExtractor()

X_seq = extractor.transform(X_raw)

print("Before:", X_raw.shape)
print("After:", X_seq.shape)

Before: (139798, 30, 24)
After: (139798, 30, 96)


split

In [11]:
import numpy as np


unique_engines = np.sort(np.unique(engine_ids))


val_ratio = 0.2
n_val = int(len(unique_engines) * val_ratio)


val_engines = set(unique_engines[-n_val:])

# mask
train_mask = np.array([eid not in val_engines for eid in engine_ids])
val_mask = ~train_mask

# split
X_train = X_seq[train_mask]
X_val = X_seq[val_mask]

y_train = y_cls[train_mask]
y_val = y_cls[val_mask]

In [12]:
print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (109009, 30, 96)
Val: (30789, 30, 96)


In [13]:
print(set(engine_ids[train_mask]) & set(engine_ids[val_mask]))

set()


Normalization

In [14]:

mean = X_train.mean(axis=(0,1))
std = X_train.std(axis=(0,1)) + 1e-8


X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

In [15]:
print(X_train.mean(), X_train.std())

-0.00823913 1.000936


model

In [16]:
import torch
import numpy as np
import random

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [17]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,   # 96
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.1
        )

        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        out, _ = self.lstm(x)        # (batch, 30, 128)
        out = out[:, -1, :]          
        out = self.fc(out)           # (batch, 2)
        return out

In [18]:
model = LSTMClassifier(input_dim=96)

Tensor + DataLoader

In [19]:
import torch
from torch.utils.data import TensorDataset, DataLoader


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)

# dataset
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

In [20]:
import torch
from torch.utils.data import TensorDataset, DataLoader

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=g
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [21]:
import torch
import numpy as np

# class distribution
#class_counts = np.bincount(y_train)

# weights
#class_weights = 1. / (class_counts + 1e-8)
#class_weights[1] *= 2 #1,5

# to tensor
#class_weights = torch.tensor(class_weights, dtype=torch.float32)

# device
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#class_weights = class_weights.to(device)

# loss
#criterion = nn.CrossEntropyLoss(weight=class_weights)

# optimizer
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
class_counts  = np.bincount(y_train)
scale         = class_counts[0] / class_counts[1]
class_weights = torch.tensor([1.0, scale], dtype=torch.float32)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-3)

print(f"Scale pos weight: {scale:.2f}")
print(f"Device: {device}")

Scale pos weight: 5.19
Device: cuda


training

In [22]:
#best_loss = float('inf')
##patience = 3
#counter = 0

#for epoch in range(30):
    # =====================
    # Training
    # =====================
   # model.train()
   # total_loss = 0

   # for X_batch, y_batch in train_loader:
    ##    outputs = model(X_batch)
        #loss = criterion(outputs, y_batch)

       # optimizer.zero_grad()
       # loss.backward()
        #optimizer.step()

       # total_loss += loss.item()

    #print(f"Epoch {epoch+1}, Train Loss: {total_loss:.4f}")

    # =====================
    # Validation
    # =====================
    ##model.eval()
    #val_loss = 0

    #with torch.no_grad():
       # for X_batch, y_batch in val_loader:
         #   outputs = model(X_batch)
          #  loss = criterion(outputs, y_batch)
          #  val_loss += loss.item()

    #print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}")

    # =====================
    # Early Stopping (on VAL loss)
    # =====================
    #if val_loss < best_loss:
       # best_loss = val_loss
        #counter = 0
    #else:
       # counter += 1

    #if counter >= patience:
        #print("Early stopping triggered ")
       # break
from sklearn.metrics import f1_score

best_f1, best_val_loss = 0, float('inf')
patience, patience_counter = 8, 0

for epoch in range(50):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output    = model(X_batch)
            val_loss += criterion(output, y_batch).item()
            all_preds.extend(output.argmax(dim=1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    avg_val_loss = val_loss / len(val_loader)
    f1 = f1_score(all_labels, all_preds)
    
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_model.pt')
    
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {f1:.4f}")

print(f"\nBest F1: {best_f1:.4f}")

Epoch 10 | Train Loss: 0.1303 | Val Loss: 0.1262 | Val F1: 0.8306
Epoch 20 | Train Loss: 0.1130 | Val Loss: 0.1148 | Val F1: 0.8566
Early stopping at epoch 22

Best F1: 0.8566


Evaluation

In [23]:
import numpy as np 
from sklearn.metrics import classification_report
import torch.nn.functional as F

thresholds = [0.4] 
model.eval()

with torch.no_grad():
    all_probs = []
    all_labels = []
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)  
        outputs = model(X_batch)
        probs = F.softmax(outputs, dim=1)
        all_probs.extend(probs[:, 1].cpu().numpy()) 
        all_labels.extend(y_batch.numpy())

for t in thresholds:
    preds = (np.array(all_probs) > t).astype(int)
    print(f"\n=== Threshold = {t} ===")
    print(classification_report(all_labels, preds))


=== Threshold = 0.4 ===
              precision    recall  f1-score   support

           0       0.99      0.94      0.97     26418
           1       0.73      0.95      0.82      4371

    accuracy                           0.94     30789
   macro avg       0.86      0.94      0.89     30789
weighted avg       0.95      0.94      0.95     30789



In [24]:
print(f"Number of labels: {len(all_labels)}")
print(f"Number of predictions: {len(preds)}")

Number of labels: 30789
Number of predictions: 30789


In [25]:
import numpy as np
import torch
all_labels = []
all_preds = []
model.eval()

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)  
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_labels.extend(y_batch.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

print(f"Number of labels: {len(all_labels)}")
print(f"Number of predictions: {len(all_preds)}")

from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))

Number of labels: 30789
Number of predictions: 30789
              precision    recall  f1-score   support

           0       0.99      0.95      0.97     26418
           1       0.75      0.93      0.83      4371

    accuracy                           0.95     30789
   macro avg       0.87      0.94      0.90     30789
weighted avg       0.95      0.95      0.95     30789



test

In [26]:
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, roc_auc_score

# Step 1: Load test data
test_path   = "/kaggle/input/datasets/christinezakaria/predictive-maintenance-data4/All_test_data.xlsx"
test_sheets = pd.read_excel(test_path, sheet_name=None)

frames = []
uid_offset = 0
for sheet_name, df_t in test_sheets.items():
    df_t = df_t.copy()
    df_t["unit_id"] = df_t["unit_id"] + uid_offset
    df_t["dataset"] = sheet_name
    uid_offset      += df_t["unit_id"].max()
    frames.append(df_t)

test = pd.concat(frames, ignore_index=True)
test = test.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
print(f"Test engines: {test['unit_id'].nunique()}")

# Step 2: Windowing بدون label
feature_cols = [col for col in test.columns
                if col.startswith("s") or col.startswith("op")]

X_test_windows = []
test_engine_ids = []

for uid, grp in test.groupby("unit_id"):
    grp  = grp.sort_values("cycle")
    vals = grp[feature_cols].values
    if len(vals) >= 30:
        X_test_windows.append(vals[-30:])
        test_engine_ids.append(uid)

X_test_raw      = np.array(X_test_windows, dtype=np.float32)
test_engine_ids = np.array(test_engine_ids)
print(f"Test windows: {X_test_raw.shape}")

# Step 3: Feature Engineering
X_test_seq = extractor.transform(X_test_raw)

# Step 4: Normalization
X_test_seq = (X_test_seq - mean) / std

# Step 5: Predictions
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
test_loader   = torch.utils.data.DataLoader(X_test_tensor, batch_size=64)

model.eval()
all_probs = []
with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs   = F.softmax(outputs, dim=1)
        all_probs.extend(probs[:, 1].cpu().numpy())

all_probs  = np.array(all_probs)
last_preds = (all_probs > 0.4).astype(int)

# Step 6: RUL 
rul_path   = "/kaggle/input/datasets/christinezakaria/predictive-maintenance-data4/RUL_data.xlsx"
rul_sheets = pd.read_excel(rul_path, sheet_name=None)

rul_list = []
for name, df_r in rul_sheets.items():
    rul_list.extend(df_r.values.flatten())

rul_values = np.array(rul_list)


valid_engine_ids = list(test_engine_ids)
all_engines      = sorted(test['unit_id'].unique())
valid_indices    = [i for i, uid in enumerate(all_engines) if uid in valid_engine_ids]

y_test_true = (rul_values[valid_indices] <= 30).astype(int)

print(f"Engines      : {len(last_preds)}")
print(f"True At Risk : {y_test_true.sum()} ({y_test_true.mean():.1%})")
print(f"Pred At Risk : {last_preds.sum()} ({last_preds.mean():.1%})")
print()
print("=" * 45)
print(classification_report(y_test_true, last_preds,
      target_names=['Healthy', 'At Risk']))
print(f"ROC-AUC: {roc_auc_score(y_test_true, all_probs):.4f}")
print("=" * 45)

Test engines: 707
Test windows: (690, 30, 24)
Engines      : 690
True At Risk : 159 (23.0%)
Pred At Risk : 194 (28.1%)

              precision    recall  f1-score   support

     Healthy       0.99      0.93      0.96       531
     At Risk       0.80      0.97      0.88       159

    accuracy                           0.94       690
   macro avg       0.90      0.95      0.92       690
weighted avg       0.95      0.94      0.94       690

ROC-AUC: 0.9906
